In [1]:
import pandas as pd
import dai

def main(datasources, start_date, end_date):
    """
    VW v5.1 (deepseek) – Rank‑Normalised Output for Maximum ModelScore
    Exact same alpha logic, only the final scaling is changed to cross‑sectional percentile.
    """
    bar1m = datasources["bar1m"]

    LOOKBACK_DAYS = 20
    query_start_date = (pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d')

    sql = f"""
    WITH minute_calc AS (
        SELECT
            date,
            instrument,
            strftime(date, '%Y-%m-%d') AS trading_day,
            strftime(date, '%H:%M:%S') AS time_str,
            (close - open) / NULLIF(open, 0) AS min_ret,
            volume,
            amount
        FROM {bar1m}
        WHERE open > 0 AND volume > 0 AND amount > 0
    ),
    daily_vol AS (
        SELECT
            trading_day,
            instrument,
            STDDEV_SAMP(min_ret) AS day_vol
        FROM minute_calc
        GROUP BY trading_day, instrument
    ),
    minute_soft AS (
        SELECT
            m.trading_day,
            m.instrument,
            m.volume,
            m.amount,
            m.time_str,
            CASE
                WHEN m.min_ret > (1.2 + 0.6 * LEAST(v.day_vol / 0.012, 1.5)) * COALESCE(v.day_vol, 0.01)
                    THEN (1.2 + 0.6 * LEAST(v.day_vol / 0.012, 1.5)) * COALESCE(v.day_vol, 0.01) + 0.25 * (m.min_ret - (1.2 + 0.6 * LEAST(v.day_vol / 0.012, 1.5)) * COALESCE(v.day_vol, 0.01))
                WHEN m.min_ret < -(1.2 + 0.6 * LEAST(v.day_vol / 0.012, 1.5)) * COALESCE(v.day_vol, 0.01)
                    THEN -(1.2 + 0.6 * LEAST(v.day_vol / 0.012, 1.5)) * COALESCE(v.day_vol, 0.01) + 0.25 * (m.min_ret + (1.2 + 0.6 * LEAST(v.day_vol / 0.012, 1.5)) * COALESCE(v.day_vol, 0.01))
                ELSE m.min_ret
            END AS min_ret_soft,
            CASE
                WHEN m.time_str >= '14:30:00' THEN 1.35
                WHEN m.time_str >= '14:00:00' THEN 1.20
                WHEN m.time_str <= '09:45:00' THEN 1.15
                WHEN m.time_str <= '10:30:00' THEN 1.05
                ELSE 0.90
            END AS tod_w
        FROM minute_calc m
        LEFT JOIN daily_vol v
            ON m.trading_day = v.trading_day AND m.instrument = v.instrument
    ),
    daily_aggregated AS (
        SELECT
            trading_day,
            instrument,
            SUM(tod_w * min_ret_soft * SQRT(volume * amount))
                / NULLIF(SUM(tod_w * SQRT(volume * amount)), 0) AS vw_ret
        FROM minute_soft
        GROUP BY trading_day, instrument
    ),
    daily_with_lag AS (
        SELECT
            *,
            AVG(vw_ret) OVER (
                PARTITION BY instrument
                ORDER BY trading_day
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) AS vw_ret_ma3,
            LAG(vw_ret, 1) OVER (PARTITION BY instrument ORDER BY trading_day) AS prev_vw
        FROM daily_aggregated
    ),
    final_raw AS (
        SELECT
            trading_day,
            instrument,
            -1.0 * (
                0.65 * TANH(30.0 * COALESCE(vw_ret_ma3, vw_ret)) +
                0.35 * TANH(40.0 * (vw_ret - COALESCE(prev_vw, 0)))
            ) AS vw5_raw
        FROM daily_with_lag
        WHERE trading_day >= '{start_date[:10]}'
          AND trading_day <= '{end_date[:10]}'
    )
    SELECT
        CAST(trading_day AS DATETIME) AS date,
        instrument,
        PERCENT_RANK() OVER (PARTITION BY trading_day ORDER BY vw5_raw) - 0.5 AS factor
    FROM final_raw
    """

    df = dai.query(sql, filters={'date': [query_start_date, end_date]}, compression=True).df()

    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()

    df = pd.merge(df, stk_pool, how='inner', on=['date', 'instrument'])
    return df

if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m'}
    start_date = '2019-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    logger.info(f"计算因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )

[2026-08-04 15:10:24] [info     ] 计算因子，区间：2019-01-01 00:00:00 ~ 2024-12-31 23:59:59
